# 24. 사전학습 Segmentation 모델 실습

이 노트북은 `23_Mask_RCNN_핵심_아이디어.ipynb` 다음 단계로, 사전학습 segmentation 모델의 출력 형식을 읽고 시각화하는 실습입니다.

실제 프로젝트에서는 segmentation 모델을 처음부터 학습시키기보다, 사전학습 모델로 inference를 먼저 해 보며 입력, 출력, 후처리 흐름을 익히는 경우가 많습니다.

이번 노트북의 목표는 다음과 같습니다.

- torchvision의 semantic segmentation 모델과 instance segmentation 모델을 구분합니다.
- DeepLabV3 출력의 class map을 읽는 방법을 익힙니다.
- Mask R-CNN 출력의 box, score, label, mask를 읽는 방법을 익힙니다.
- 사전학습 가중치가 없는 환경에서도 같은 결과 형식을 예제값으로 연습합니다.

## 24-1. 준비

이 노트북은 자동으로 패키지를 설치하거나 가중치를 다운로드하지 않습니다. `torchvision`과 사전학습 가중치가 로컬 캐시에 있으면 실제 모델을 사용할 수 있고, 그렇지 않으면 예제 출력값으로 같은 후처리 흐름을 진행합니다.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw

available_fonts = {f.name for f in fm.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        plt.rcParams['font.family'] = font_name
        break

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.unicode_minus'] = False
np.random.seed(42)

## 24-2. 실습 이미지 준비

외부 이미지 파일 없이도 실행할 수 있도록 간단한 예제 이미지를 만듭니다. 실제 사진으로 바꾸려면 `image_path`에 본인의 이미지 경로를 넣으면 됩니다.

In [ ]:
image_path = None  # 예: 'data/my_street.jpg'

if image_path is not None and Path(image_path).exists():
    image = Image.open(image_path).convert('RGB')
else:
    image = Image.new('RGB', (360, 240), color=(185, 215, 235))
    draw = ImageDraw.Draw(image)
    draw.rectangle((0, 165, 360, 240), fill=(115, 115, 115))
    draw.rectangle((40, 95, 135, 165), fill=(230, 170, 60))
    draw.ellipse((185, 65, 245, 155), fill=(70, 160, 85))
    draw.rectangle((265, 105, 330, 165), fill=(80, 125, 210))
    draw.rectangle((72, 130, 104, 155), fill=(40, 40, 40))
    draw.rectangle((205, 120, 225, 150), fill=(40, 40, 40))

plt.figure(figsize=(7, 4))
plt.imshow(image)
plt.title('실습 이미지')
plt.axis('off')
plt.show()

## 24-3. 사전학습 가중치 사용 가능 여부 확인

`torchvision` 모델의 `weights=DEFAULT`를 사용하면 가중치가 없을 때 다운로드를 시도할 수 있습니다. 이 노트북은 학습 환경을 건드리지 않기 위해, 로컬 캐시에 가중치 파일이 있을 때만 실제 사전학습 모델을 사용합니다.

가중치가 없다면 `use_real_models=False`로 두고 예제 결과를 사용합니다.

In [ ]:
use_real_models = False
model_status = []

def checkpoint_exists(weight_enum):
    try:
        import torch
        filename = Path(urlparse(weight_enum.url).path).name
        checkpoint_path = Path(torch.hub.get_dir()) / 'checkpoints' / filename
        return checkpoint_path.exists(), checkpoint_path
    except Exception as exc:
        return False, exc

try:
    import torch
    import torchvision
    from torchvision.transforms import functional as F
    from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights, deeplabv3_resnet50
    from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights, maskrcnn_resnet50_fpn

    deeplab_weights = DeepLabV3_ResNet50_Weights.DEFAULT
    maskrcnn_weights = MaskRCNN_ResNet50_FPN_Weights.DEFAULT
    deeplab_ready, deeplab_path = checkpoint_exists(deeplab_weights)
    maskrcnn_ready, maskrcnn_path = checkpoint_exists(maskrcnn_weights)

    model_status.append(f'DeepLabV3 cached: {deeplab_ready} ({deeplab_path})')
    model_status.append(f'Mask R-CNN cached: {maskrcnn_ready} ({maskrcnn_path})')
    use_real_models = deeplab_ready and maskrcnn_ready
except Exception as exc:
    model_status.append(f'torchvision 또는 가중치 확인 불가: {exc}')

for line in model_status:
    print(line)
print('use_real_models:', use_real_models)

## 24-4. Semantic segmentation: DeepLabV3 출력 읽기

DeepLabV3 같은 semantic segmentation 모델은 보통 `out` 텐서를 출력합니다.

```text
out shape = [batch, num_classes, height, width]
```

각 픽셀에서 class score가 가장 큰 채널을 고르면 class map이 됩니다.

In [ ]:
VOC_COLORS = np.array([
    [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0], [0, 0, 128],
    [128, 0, 128], [0, 128, 128], [128, 128, 128], [64, 0, 0], [192, 0, 0],
    [64, 128, 0], [192, 128, 0], [64, 0, 128], [192, 0, 128], [64, 128, 128],
    [192, 128, 128], [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
    [0, 64, 128]
], dtype=np.uint8)

VOC_NAMES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car',
    'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

def colorize_class_map(class_map):
    safe_map = np.asarray(class_map) % len(VOC_COLORS)
    return VOC_COLORS[safe_map]


def fallback_semantic_map(width=360, height=240):
    class_map = np.zeros((height, width), dtype=np.int64)
    class_map[165:, :] = 7       # car/road 역할의 예제 영역
    class_map[65:155, 185:245] = 15
    class_map[95:165, 40:135] = 7
    class_map[105:165, 265:330] = 15
    return class_map

if use_real_models:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    deeplab = deeplabv3_resnet50(weights=deeplab_weights).to(device).eval()
    preprocess = deeplab_weights.transforms()
    tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = deeplab(tensor)['out'][0]
    semantic_map = output.argmax(0).cpu().numpy()
    semantic_source = 'DeepLabV3 실제 출력'
else:
    semantic_map = fallback_semantic_map(image.width, image.height)
    semantic_source = '예제 semantic map'

semantic_color = colorize_class_map(semantic_map)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image)
axes[0].set_title('image')
axes[1].imshow(semantic_color)
axes[1].set_title(semantic_source)
for ax in axes:
    ax.axis('off')
plt.show()

unique_ids = sorted(np.unique(semantic_map).tolist())
print('class ids:', unique_ids[:20])
print('class names:', [VOC_NAMES[i] if i < len(VOC_NAMES) else str(i) for i in unique_ids[:20]])

## 24-5. Semantic segmentation 결과를 이미지 위에 겹치기

class map은 단독으로 볼 수도 있고, 원본 이미지 위에 반투명하게 겹쳐서 볼 수도 있습니다. 실제 분석에서는 원본과 mask overlay를 함께 확인하는 습관이 중요합니다.

In [ ]:
image_np = np.asarray(image).astype(np.float32)
overlay = (image_np * 0.55 + semantic_color.astype(np.float32) * 0.45).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title('image')
axes[1].imshow(semantic_color)
axes[1].set_title('class map')
axes[2].imshow(overlay)
axes[2].set_title('overlay')
for ax in axes:
    ax.axis('off')
plt.show()

## 24-6. Instance segmentation: Mask R-CNN 출력 읽기

Mask R-CNN 계열 모델의 출력은 이미지마다 dictionary 형태로 나오는 경우가 많습니다.

```text
boxes   : [N, 4]
labels  : [N]
scores  : [N]
masks   : [N, 1, H, W]
```

여기서 `N`은 탐지된 instance 개수입니다. score threshold를 적용해 신뢰도가 낮은 instance를 제거한 뒤, mask threshold를 적용해 binary mask로 바꿉니다.

In [ ]:
COCO_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train',
    'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench',
    'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe',
    'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
    'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard',
    'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet',
    'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven',
    'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush'
]


def fallback_instances(width=360, height=240):
    masks = []
    yy, xx = np.mgrid[:height, :width]

    person1 = (((xx - 215) / 30) ** 2 + ((yy - 110) / 45) ** 2 <= 1).astype(float)
    person2 = np.zeros((height, width), dtype=float)
    person2[105:165, 265:330] = 1.0
    car = np.zeros((height, width), dtype=float)
    car[95:165, 40:135] = 1.0

    masks.extend([person1, person2, car])
    return [
        {'label': 1, 'score': 0.93, 'box': (185, 65, 245, 155), 'mask': masks[0]},
        {'label': 1, 'score': 0.86, 'box': (265, 105, 330, 165), 'mask': masks[1]},
        {'label': 3, 'score': 0.81, 'box': (40, 95, 135, 165), 'mask': masks[2]},
        {'label': 1, 'score': 0.22, 'box': (10, 10, 50, 70), 'mask': np.zeros((height, width), dtype=float)},
    ]

if use_real_models:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    mask_model = maskrcnn_resnet50_fpn(weights=maskrcnn_weights).to(device).eval()
    tensor = F.to_tensor(image).to(device)
    with torch.no_grad():
        result = mask_model([tensor])[0]
    instances = []
    for box, label, score, mask in zip(result['boxes'], result['labels'], result['scores'], result['masks']):
        instances.append({
            'label': int(label.cpu().item()),
            'score': float(score.cpu().item()),
            'box': tuple(box.cpu().numpy().tolist()),
            'mask': mask[0].cpu().numpy(),
        })
    instance_source = 'Mask R-CNN 실제 출력'
else:
    instances = fallback_instances(image.width, image.height)
    instance_source = '예제 instance 출력'

score_threshold = 0.5
selected = [inst for inst in instances if inst['score'] >= score_threshold]

print(instance_source)
print('selected instances:', len(selected))
for inst in selected:
    label = inst['label']
    name = COCO_NAMES[label] if label < len(COCO_NAMES) else str(label)
    box = tuple(round(v, 1) for v in inst['box'])
    print(f'{name:<12} score={inst["score"]:.2f} box={box}')

## 24-7. Instance mask 시각화

instance segmentation 결과는 각 객체 mask를 서로 다른 색으로 칠해 보면 읽기 쉽습니다. 실제 분석에서는 score threshold와 mask threshold를 함께 조정하면서 결과가 어떻게 달라지는지 확인합니다.

In [ ]:
INSTANCE_COLORS = np.array([
    [84, 162, 75],
    [245, 133, 24],
    [229, 87, 86],
    [114, 183, 178],
    [177, 122, 161],
], dtype=np.uint8)

mask_threshold = 0.5
instance_overlay = np.asarray(image).astype(np.float32).copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(image)
for i, inst in enumerate(selected):
    color = INSTANCE_COLORS[i % len(INSTANCE_COLORS)]
    binary_mask = inst['mask'] >= mask_threshold
    instance_overlay[binary_mask] = instance_overlay[binary_mask] * 0.45 + color * 0.55

    x1, y1, x2, y2 = inst['box']
    label = inst['label']
    name = COCO_NAMES[label] if label < len(COCO_NAMES) else str(label)
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color / 255, linewidth=2))
    ax.text(x1, max(0, y1 - 4), f'{name} {inst["score"]:.2f}', color=color / 255, fontsize=10, weight='bold')

ax.imshow(instance_overlay.astype(np.uint8), alpha=0.75)
ax.set_title('instance mask overlay')
ax.axis('off')
plt.show()

## 24-8. DeepLabV3와 Mask R-CNN 비교

두 모델은 모두 segmentation을 다루지만 출력의 의미가 다릅니다.

- DeepLabV3: 픽셀마다 class를 예측합니다. 같은 class의 객체는 하나의 영역처럼 보일 수 있습니다.
- Mask R-CNN: 객체 후보마다 class, box, mask를 예측합니다. 같은 class의 객체도 instance별로 분리됩니다.

문제가 `도로 영역`, `하늘 영역`, `피부 영역`처럼 클래스별 영역을 찾는 일이라면 semantic segmentation이 자연스럽습니다. 문제가 `사람 몇 명`, `차량 각각의 윤곽`, `개별 물체 crop`처럼 instance 단위라면 Mask R-CNN이 더 적합합니다.

In [ ]:
summary = [
    ('DeepLabV3', 'semantic segmentation', 'class map', '[C, H, W]'),
    ('Mask R-CNN', 'instance segmentation', 'boxes + labels + scores + masks', 'N개 instance'),
]

for model_name, task, output, shape in summary:
    print(f'{model_name:<12} | {task:<24} | {output:<32} | {shape}')

## 24-9. 실제 사진으로 실험하기

실제 사진을 사용하려면 위쪽의 `image_path` 값을 바꿉니다.

```python
image_path = 'data/my_image.jpg'
```

사전학습 모델을 실제로 사용하려면 `torch`, `torchvision`이 설치되어 있고, 해당 가중치가 로컬 캐시에 있어야 합니다. 가중치가 없다면 인터넷이 되는 환경에서 한 번 다운로드하거나, 체크포인트 파일을 torch hub cache의 `checkpoints` 폴더에 직접 둘 수 있습니다.

이 노트북의 핵심은 모델을 무조건 실행하는 것이 아니라, segmentation 모델의 출력 형식을 정확히 읽고 후처리하는 흐름을 익히는 것입니다.

## 정리

- DeepLabV3 같은 semantic segmentation 모델은 픽셀별 class map을 출력합니다.
- Mask R-CNN 같은 instance segmentation 모델은 instance별 box, label, score, mask를 출력합니다.
- 실제 모델이 없어도 같은 출력 형식을 예제값으로 만들어 후처리와 시각화를 연습할 수 있습니다.
- segmentation 결과를 볼 때는 원본 이미지, class map, mask overlay를 함께 확인하는 것이 좋습니다.

이로써 3장은 `detection -> semantic segmentation -> FCN -> U-Net -> instance segmentation -> Mask R-CNN -> 사전학습 모델 실습` 흐름으로 마무리됩니다.